# Simulazione Demografica dell'Italia (2025 – 2050)
## Modello per Componenti (Cohort-Component Model) applicato alla Discendenza Ancestrale e all'Origine Geografica

---

### 1. Abstract e Obiettivi del Progetto
Le proiezioni demografiche ufficiali (Istat, Eurostat) classificano la popolazione residente esclusivamente in base alla **cittadinanza formale/giuridica** (*italiani* vs *stranieri*). Poiché ogni anno l'ordinamento italiano concede la cittadinanza a oltre 200.000 residenti stranieri, le statistiche anagrafiche convenzionali "assorbono" costantemente le persone naturalizzate e le seconde generazioni all'interno della voce *Italia*, rendendo invisibile la reale dinamica etno-demografica e l'evoluzione della discendenza nel Paese.

**L'obiettivo di questo modello** è costruire una simulazione demografica previsionale per il periodo **2025 – 2050** che tracci la popolazione residente in base all'**effettiva radice/discendenza geografica**, separando con rigore contabile:
1. La **popolazione autoctona di discendenza italiana** (depurata delle naturalizzazioni recenti);
2. Le **singole comunità di origine estera** (comprese le prime generazioni, i cittadini naturalizzati e le seconde/terze generazioni nate sul territorio nazionale).

Il motore di calcolo implementa un **Cohort-Component Model** multi-popolazione matriciale, guidato da dati ufficiali Istat su natalità, mortalità senile e flussi migratori internazionali.

---

### 2. Fonti dei Dati Utilizzate (Dataset Istat)
Tutti i dati di partenza derivano dalle rilevazioni demografiche ufficiali dell'Istat (annate di riferimento 2024 e 2025):

| File di Input | Tipologia di Rilevazione | Variabili Chiave Utilizzate |
| :--- | :--- | :--- |
| **`Dati_cittadinanza_2025.csv`** | Stock anagrafico al 1° gennaio 2025 | Popolazione nazionale per cittadinanza, sesso, macro-area e continente. |
| **`acquisizioni_cittadinanza.csv`** | Serie storica concessioni (2012–2024) | Concessioni di cittadinanza italiana suddivise per paese d'origine precedente. |
| **`Popolazione residente.csv`** | Piramide demografica totale (2025) | Popolazione residente nazionale per ogni singolo anno d'età (da 0 a 100+ anni) e sesso. |
| **`Popolazione straniera residente.csv`** | Piramide demografica straniera (2025) | Popolazione straniera residente per singolo anno d'età (da 0 a 100+ anni) e sesso. |
| **`Decessi2025.csv`** | Tavola di Mortalità Ufficiale (*Life Table*) | Probabilità prospettiche di sopravvivenza ($P_x$) e di morte ($q_x$) per età e sesso. |
| **`Nascite2024.csv`** | Registro degli Eventi di Nascita | Nuovi nati vivi sul territorio italiano per paese di cittadinanza dei genitori/madre. |
| **`migrazioni_italiani.xlsx` (Tav. 3)** | Bilancio migratorio internazionale globale | Totale nazionale delle iscrizioni dall'estero (451.583) e cancellazioni (188.903). |
| **`immigrati_per_paese.xlsx` (Tav. 7)** | Iscrizioni per trasferimento di residenza | Immigrati stranieri per continente e principali paesi di provenienza. |
| **`emigrati_per_paese.xlsx` (Tav. 8)** | Cancellazioni per l'estero | Emigrati stranieri per continente e principali paesi di destinazione. |

---

### 3. Pipeline Metodologica di Elaborazione

#### Fase 1: Rettifica Etnodemografica della Popolazione Base (Anno 2025)
* **Scorporo dei naturalizzati**: Attraverso la serie storica delle acquisizioni di cittadinanza (2012–2024), circa **1,78 milioni di individui** vengono sottratti dal totale giuridico dell'Italia e restituiti al loro reale paese d'origine (mantenendo la sex-ratio della comunità).
* **Chiusura contabile dei residui UE/extra-UE**: Le acquisizioni classificate come *"Altri paesi UE"* sono confluite nel blocco aggregato europeo; quelle *"extra-UE"* sono state redistribuite proporzionalmente tra Africa, Asia, America e Oceania.

#### Fase 2: Selezione dei Paesi Chiave e Accorpamento Continentale
Per evitare l'eccessiva frammentazione su oltre 200 micro-stati e mantenere robusto il modello, è stata applicata una **soglia a gradino di significatività demografica**:
* **Paesi Top ($\ge 50.000$ residenti)**: Mantengono la propria tracciatura individuale autonoma (~20 comunità principali: Romania, Albania, Marocco, Cina, Ucraina, Bangladesh, India, Pakistan, Egitto, Filippine, Tunisia, Senegal, Perù, Nigeria, ecc.).
* **Paesi Minori ($< 50.000$ residenti)**: Vengono fusi nei rispettivi contenitori macro-regionali:
  $$\text{Altro (Europa)}, \quad \text{Altro (Asia)}, \quad \text{Altro (Africa)}, \quad \text{Altro (America)}, \quad \text{Altro (Oceania)}$$

#### Fase 3: Costruzione della Matrice Base per Età e Sesso ($Paese \times Età \times Sesso$)
* **Curva Autoctona**: Calcolata per differenza puntuale ($\text{Popolazione Totale}(x) - \text{Popolazione Straniera}(x)$). Risulta una piramide rovesciata (fortemente senescente, con picco tra i 55 e i 75 anni).
* **Curva Migratoria**: Basata sul profilo per età della popolazione straniera aggregata (struttura giovane, concentrata nella fascia lavorativa 20–45 anni e nella prima infanzia 0–14 anni), scalata sui volumi e sul rapporto maschi/femmine specifico di ogni singola comunità.

---

### 4. Il Motore di Simulazione Annuale ($t \rightarrow t+1$)
A ogni ciclo temporale annuale, la popolazione subisce l'azione combinata di tre forze demografiche elementari:

#### A. Mortalità e Invecchiamento delle Coorti (*Cohort Aging*)
Ogni individuo di età $x$ (con $0 \le x \le 99$) sopravvive all'anno successivo con la probabilità $P_{x,\text{sesso}}$ derivata dalle tavole attuariali dell'Istat:
$$N_{x+1, s}(t+1) = N_{x, s}(t) \cdot P_{x, s}$$
Per la classe terminale aperta (100+ anni), i sopravvissuti della coorte 99 si sommano a quelli già centenari sopravvissuti:
$$N_{100, s}(t+1) = N_{99, s}(t) \cdot P_{99, s} + N_{100, s}(t) \cdot P_{100, s}$$

#### B. Fecondità e Nuove Coorti di Neonati (Età 0)
Per ogni gruppo etno-geografico, il volume annuo di nascite è calcolato a partire dal **Tasso Generico di Fecondità** ($GFR$) specifico del gruppo, parametrato sullo stock delle sole donne in età riproduttiva (15–49 anni):
$$\text{Nuove Nascite}(t+1) = \left( \sum_{x=15}^{49} N_{x, \text{femmine}}(t) \right) \cdot GFR_{\text{gruppo}}$$
I nuovi nati vengono ripartiti all'età 0 secondo il rapporto sessuale biologico naturale alla nascita registrato dall'Istat ($51{,}45\%$ maschi e $48{,}55\%$ femmine).  
*Nota fondamentale*: I neonati nati in Italia da madri di origine estera vengono mantenuti all'interno del gruppo della madre (tracciamento biologico-generazionale, nessuna naturalizzazione automatica applicata).

#### C. Flussi Migratori Netti e Profilo per Età
Il saldo migratorio netto annuo ($\text{Immigrati} - \text{Emigrati}$) viene attribuito a ciascun paese usando i dati 2024:
* **Italiani autoctoni**: Ricavati per differenza contabile chiusa ($\text{Saldo Totale Nazionale} - \text{Saldo Totale Stranieri} = -82.550 \text{ persone/anno}$, a rappresentare l'emigrazione netta/fuga di cervelli).
* **Paesi esteri e residui**: Assegnati con i valori reali di Tavola 7/8 Istat e ripartizione proporzionale del residuo continentale.
* **Curva di età dei migranti (Modello Rogers-Castro)**: I migranti netti vengono suddivisi per sesso in base alla sex-ratio propria del paese e distribuiti sulle età giovanili/lavorative secondo una curva a campana compresa tra i **18 e i 38 anni**, con picco empirico a **26 anni**.

---

### 5. Assunzioni e Limiti Metodologici

1. **Ipotesi di Stazionarietà delle Forze Demografiche (*Static Parameter Assumption*)**:  
   I tassi di fecondità, le probabilità di sopravvivenza e i saldi migratori netti sono mantenuti costanti ai livelli consolidati del 2024/2025. È il presupposto standard per gli scenari previsivi *"a politiche invariate"* (*baseline projections*).
2. **Assenza di Matrimoni Misti (*Endogamy Simplification*)**:  
   Il modello assegna la totalità della discendenza al gruppo di origine della madre; non include modelli stocastici di intermarriage tra autoctoni e immigrati.
3. **Assenza di Naturalizzazioni nel Continuo**:  
   Il modello non simula la transizione anagrafica formale dei cittadini stranieri verso la cittadinanza italiana nel periodo 2025–2050, isolando deliberatamente la linea di ascendenza e il background etnico-culturale.
4. **Conservazione Contabile Assoluta (*Zero Leakage Constraint*)**:  
   La somma dei flussi migratori e delle piramidi di partenza quadra esattamente all'unità con i totali censuari Istat.

---

### 6. Risultati Salienti della Simulazione (2025 vs 2050)

* **Popolazione Complessiva**:  
  L'Italia passa da **58,94 milioni** a **56,10 milioni** di residenti ($-2,84 \text{ mln}$), collocandosi con altissima precisione nell'intervallo delle proiezioni ufficiali dell'Istat (Scenario Mediano: $55,0 \text{ mln}$, Scenario Alto: $56,5 \text{ mln}$).
* **Popolazione Autoctona di Discendenza Italiana**:  
  Subisce una contrazione netta di **quasi 14 milioni di individui** (da $51,61 \text{ mln}$ a $37,62 \text{ mln}$, scendendo dall'**87,56%** al **67,06%** del totale). Questo fenomeno è guidato dall'invecchiamento e dalla naturale scomparsa delle massicce coorti nate durante il Baby Boom (1958–1968), non rimpiazzate da una natalità italiana ferma a $1{,}14$ figli per donna.
* **Popolazione di Origine Estera (1ª, 2ª e 3ª generazione)**:  
  Sale da **7,33 milioni (12,44%)** a **18,48 milioni (32,94%)**, rappresentando nel 2050 circa **1 residente su 3**, guidata da un saldo migratorio giovane e da una natalità più dinamica (con picchi di espansione per le comunità del Bangladesh, Pakistan, Egitto, Albania e Marocco).

In [33]:
# INSTALLAZIONE DIPENDENZE E IMPORT DELLE LIBRERIE

# Installazione automatica dei pacchetti necessari nel virtual environment
# - pandas: manipolazione dati tabellari
# - numpy: calcolo vettoriale per il motore di simulazione demografica
# - openpyxl: motore necessario a pandas per leggere i file Excel (.xlsx)
# - matplotlib: per la creazione di grafici e piramidi dell'età
%pip install pandas numpy openpyxl matplotlib

# Import delle librerie
import pandas as pd
import numpy as np
from IPython.display import display
import matplotlib.pyplot as plt

# Configurazione visualizzazione di Pandas (evita tagli o righe spezzate)
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_rows', 50)
pd.set_option('display.width', 1000)

print("Setup completato con successo: tutte le librerie sono pronte all'uso!")

Note: you may need to restart the kernel to use updated packages.
Setup completato con successo: tutte le librerie sono pronte all'uso!



[notice] A new release of pip available: 22.3 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [34]:
# CARICAMENTO DATI CITTADINANZA 2025

# Lettura del dataset Istat della popolazione per paese di cittadinanza

df_raw = pd.read_csv("Dati_cittadinanza_2025.csv", sep=";")
#print(df_raw.head())

C:\Users\finot\AppData\Local\Temp\ipykernel_1780\2034328687.py:5: DtypeWarning: Columns (1: Codice Istat) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw = pd.read_csv("Dati_cittadinanza_2025.csv", sep=";")


In [35]:
# FILTRO LIVELLO NAZIONALE

# Estrazione delle sole righe relative all'intero territorio italiano
df_ita = df_raw[df_raw['Denominazione'].astype(str).str.strip().str.lower() == 'italia'].copy()

print(df_ita.head())

        Anno Codice Istat Denominazione  Codice stato di cittadinanza Stato di cittadinanza                     Zona Continente    Maschi   Femmine    Totale
267171  2025           IT        Italia                           100                Italia           Unione Europea     Europa  26182095  27390118  53572213
267172  2025           IT        Italia                           301           Afghanistan  Asia Centro-Meridionale       Asia     13663      4132     17795
267173  2025           IT        Italia                           201               Albania  Europa Centro-Orientale     Europa    213316    201306    414622
267174  2025           IT        Italia                           401               Algeria    Africa Settentrionale     Africa     13088      6663     19751
267175  2025           IT        Italia                           202               Andorra      Altri paesi europei     Europa         3         6         9


In [36]:
# SELEZIONE COLONNE E PULIZIA DATI

# Mantenimento delle colonne anagrafiche essenziali
colonne_utili = ['Stato di cittadinanza', 'Continente', 'Maschi', 'Femmine', 'Totale']
df_ita = df_ita[colonne_utili].copy()

# Pulizia degli spazi nei testi e gestione dei valori numerici mancanti
df_ita['Stato di cittadinanza'] = df_ita['Stato di cittadinanza'].astype(str).str.strip()
df_ita['Continente'] = df_ita['Continente'].astype(str).str.strip()

#Gestione dei dati mancanti
for col in ['Maschi', 'Femmine', 'Totale']:
    df_ita[col] = pd.to_numeric(df_ita[col], errors='coerce').fillna(0)

print(df_ita.head())

       Stato di cittadinanza Continente    Maschi   Femmine    Totale
267171                Italia     Europa  26182095  27390118  53572213
267172           Afghanistan       Asia     13663      4132     17795
267173               Albania     Europa    213316    201306    414622
267174               Algeria     Africa     13088      6663     19751
267175               Andorra     Europa         3         6         9


In [37]:
# MAPPA CONTINENTI

# Dizionario di corrispondenza per risalire rapidamente al continente di ogni stato
mappa_paese_continente = df_ita.set_index('Stato di cittadinanza')['Continente'].to_dict()
print(mappa_paese_continente)

{'Italia': 'Europa', 'Afghanistan': 'Asia', 'Albania': 'Europa', 'Algeria': 'Africa', 'Andorra': 'Europa', 'Angola': 'Africa', 'Antigua e Barbuda': 'America', 'Apolide': nan, 'Arabia Saudita': 'Asia', 'Argentina': 'America', 'Armenia': 'Asia', 'Australia': 'Oceania', 'Austria': 'Europa', 'Azerbaigian': 'Asia', 'Bahamas': 'America', 'Bahrein': 'Asia', 'Bangladesh': 'Asia', 'Barbados': 'America', 'Belgio': 'Europa', 'Belize': 'America', 'Benin': 'Africa', 'Bhutan': 'Asia', 'Bielorussia': 'Europa', 'Bolivia': 'America', 'Bosnia-Erzegovina': 'Europa', 'Botswana': 'Africa', 'Brasile': 'America', 'Brunei Darussalam': 'Asia', 'Bulgaria': 'Europa', 'Burkina Faso': 'Africa', 'Burundi': 'Africa', 'Cambogia': 'Asia', 'Camerun': 'Africa', 'Canada': 'America', 'Capo Verde': 'Africa', 'Ciad': 'Africa', 'Cile': 'America', 'Cina': 'Asia', 'Cipro': 'Europa', 'Colombia': 'America', 'Comore': 'Africa', 'Congo': 'Africa', 'Corea del Nord': 'Asia', 'Corea del Sud': 'Asia', "Costa d'Avorio": 'Africa', 'Cost

In [38]:
# SEPARAZIONE ITALIANI E STRANIERI

# Isolamento della popolazione con cittadinanza italiana dai residenti stranieri
df_italiani = df_ita[df_ita['Stato di cittadinanza'] == 'Italia'].copy()
df_stranieri = df_ita[df_ita['Stato di cittadinanza'] != 'Italia'].copy()

print(df_italiani)
print("="*80)
print(df_stranieri.head(15)) #primi 15 stati in ordine alfabetico

       Stato di cittadinanza Continente    Maschi   Femmine    Totale
267171                Italia     Europa  26182095  27390118  53572213
       Stato di cittadinanza Continente  Maschi  Femmine  Totale
267172           Afghanistan       Asia   13663     4132   17795
267173               Albania     Europa  213316   201306  414622
267174               Algeria     Africa   13088     6663   19751
267175               Andorra     Europa       3        6       9
267176                Angola     Africa     593      595    1188
267177     Antigua e Barbuda    America      13       12      25
267178               Apolide        NaN     307      218     525
267179        Arabia Saudita       Asia      97       60     157
267180             Argentina    America    8292     9812   18104
267181               Armenia       Asia     693      898    1591
267182             Australia    Oceania     668     1058    1726
267183               Austria     Europa    2132     4488    6620
267184         

In [39]:
# ORDINAMENTO POPOLAZIONE STRANIERA

# Classifica decrescente delle comunità estere per consistenza numerica complessiva
df_stranieri = df_stranieri.sort_values(by='Totale', ascending=False).reset_index(drop=True)
print(df_stranieri.head(20))

   Stato di cittadinanza Continente  Maschi  Femmine   Totale
0                Romania     Europa  460358   592684  1053042
1                Albania     Europa  213316   201306   414622
2                Marocco     Africa  226134   186323   412457
3                   Cina       Asia  156794   154456   311250
4                Ucraina     Europa   72075   215112   287187
5             Bangladesh       Asia  158337    55285   213622
6                 Egitto     Africa  119163    54978   174141
7                  India       Asia   97029    74400   171429
8               Pakistan       Asia  127309    43870   171179
9              Filippine       Asia   66182    87273   153455
10               Nigeria     Africa   75646    56483   132129
11               Tunisia     Africa   78434    45394   123828
12               Senegal     Africa   86994    32073   119067
13                  Perù    America   49810    66359   116169
14             Sri Lanka       Asia   59392    54313   113705
15      

In [40]:
# CARICAMENTO ACQUISIZIONI DI CITTADINANZA (2012-2024)

# Lettura della serie storica con gestione del separatore delle migliaia e filtro Italia
df_acq = pd.read_csv("acquisizioni_cittadinanza.csv", sep=",", thousands=".")
df_acq_italia = df_acq[df_acq['Territorio'].astype(str).str.strip().str.lower() == 'italia'].copy()
print(df_acq_italia)

    Anno Territorio  Albania  Romania  Ucraina  Macedonia del Nord  Moldova  Kosovo  Bangladesh  Filippine  India  Pakistan  Egitto  Ghana  Marocco  Nigeria  Senegal  Tunisia  Argentina  Brasile  Ecuador  Perù  Altri paesi UE  Altri paesi extra-UE  Totale
5   2012     Italia     9493     3272     1580                1219     1222     513        1460        894   2366      1522    1342   1288    14728      938     1070     2555        332     1442      677  1589            2051                 13830   65383
11  2013     Italia    13671     4386     1806                2089     1430    1348        3511       1048   4863      3532    2130   2838    25421     1611     2263     3521        362     1786      854  2055            2293                 17894  100712
17  2014     Italia    21148     6442     1443                2847     1475    1928        5323       1937   5015      4216    3138   3700    29025     2217     4037     4411        331     1579     1182  3136            2984       

In [41]:
# SOMMA ACQUISIZIONI PER PAESE D'ORIGINE

# Calcolo del totale storico delle concessioni di cittadinanza per ciascuna nazionalità
col_paesi = [c for c in df_acq_italia.columns if c not in ['Anno', 'Territorio']]
for col in col_paesi:
    df_acq_italia[col] = pd.to_numeric(df_acq_italia[col], errors='coerce').fillna(0)
tot_acq = df_acq_italia[col_paesi].sum()
print(tot_acq)


Albania                  343452
Romania                  132613
Ucraina                   39603
Macedonia del Nord        48958
Moldova                   55553
Kosovo                    29156
Bangladesh                68004
Filippine                 31981
India                     86849
Pakistan                  59189
Egitto                    49413
Ghana                     31601
Marocco                  311891
Nigeria                   31232
Senegal                   46519
Tunisia                   50751
Argentina                 51852
Brasile                   91219
Ecuador                   40657
Perù                      47516
Altri paesi UE            37751
Altri paesi extra-UE     273968
Totale                  1959728
dtype: int64


In [42]:
# PREPARAZIONE STRUTTURA PER LA RETTIFICA

# Impostazione del paese come indice per consentire le operazioni di sottrazione e aggiunta
df_completo = df_ita.set_index('Stato di cittadinanza')
print(df_completo)

                      Continente    Maschi   Femmine    Totale
Stato di cittadinanza                                         
Italia                    Europa  26182095  27390118  53572213
Afghanistan                 Asia     13663      4132     17795
Albania                   Europa    213316    201306    414622
Algeria                   Africa     13088      6663     19751
Andorra                   Europa         3         6         9
...                          ...       ...       ...       ...
Venezuela                America      6533     10448     16981
Vietnam                     Asia       521      1268      1789
Yemen                       Asia       213       134       347
Zambia                    Africa       116       137       253
Zimbabwe                  Africa       117       156       273

[196 rows x 4 columns]


In [43]:
# VOLUME TOTALE NATURALIZZATI DA RIALLOCARE

# Estrazione del numero complessivo di cittadini da sottrarre dalla voce Italia
tot_naturalizzati = tot_acq['Totale'].astype(float)
print(tot_naturalizzati)

1959728.0


In [44]:
# PROPORZIONE DI GENERE DEGLI ITALIANI

# Calcolo della quota maschile per ripartire la sottrazione mantenendo il rapporto tra i sessi
p_m_ita = df_completo.loc['Italia', 'Maschi'] / df_completo.loc['Italia', 'Totale']
print(p_m_ita)
df_completo.loc['Italia', 'Maschi'] = df_completo.loc['Italia', 'Maschi'].astype(float)

0.4887252837585784


In [45]:
# PULIZIA INTESTAZIONI E FILTRO NAZIONALE ACQUISIZIONI

# Rimozione spazi bianchi dalle colonne e selezione del solo livello nazionale
df_acq.columns = df_acq.columns.astype(str).str.strip()
df_acq_italia = df_acq[df_acq['Territorio'].astype(str).str.strip().str.lower() == 'italia'].copy()

# CALCOLO ACQUISIZIONI STORICHE PER PAESE

# Conversione numerica e somma delle concessioni di cittadinanza per ciascun paese d'origine
col_paesi = [c for c in df_acq_italia.columns if c not in ['Anno', 'Territorio']]
for col in col_paesi:
    df_acq_italia[col] = pd.to_numeric(df_acq_italia[col], errors='coerce').fillna(0)

tot_acq = df_acq_italia[col_paesi].sum()

print(f"Totale persone naturalizzate da riallocare: {int(tot_acq['Totale']):,}".replace(',', '.'))

Totale persone naturalizzate da riallocare: 1.959.728


In [46]:
# PREPARAZIONE STRUTTURA PER OPERAZIONI NUMERICHE

# Impostazione del paese come indice e conversione in decimale per evitare conflitti di tipo
df_completo = df_ita.copy().set_index('Stato di cittadinanza')
df_completo[['Maschi', 'Femmine', 'Totale']] = df_completo[['Maschi', 'Femmine', 'Totale']].astype(float)

# SOTTRAZIONE DEI NATURALIZZATI DALLA VOCE ITALIA

# Scorporo dei nuovi cittadini mantenendo invariata la proporzione tra i sessi
totale_da_togliere = float(tot_acq['Totale'])
pm_ita = df_completo.loc['Italia', 'Maschi'] / df_completo.loc['Italia', 'Totale']

df_completo.loc['Italia', 'Totale'] -= totale_da_togliere
df_completo.loc['Italia', 'Maschi'] -= (totale_da_togliere * pm_ita)
df_completo.loc['Italia', 'Femmine'] -= (totale_da_togliere * (1 - pm_ita))

# ALLOCAZIONE DEI NATURALIZZATI AI RISPETTIVI PAESI D'ORIGINE

# Incremento delle consistenze dei paesi esteri secondo le cittadinanze concesse
paesi_singoli = [c for c in col_paesi if c not in ['Totale', 'Altri paesi UE', 'Altri paesi extra-UE']]

for p in paesi_singoli:
    if p in df_completo.index and tot_acq[p] > 0:
        val = float(tot_acq[p])
        tot_attuale = df_completo.loc[p, 'Totale']
        pm = (df_completo.loc[p, 'Maschi'] / tot_attuale) if tot_attuale > 0 else 0.5
        
        df_completo.loc[p, 'Totale'] += val
        df_completo.loc[p, 'Maschi'] += val * pm
        df_completo.loc[p, 'Femmine'] += val * (1 - pm)

# CALCOLO QUOTE PERCENTUALI E CUMULATA SULL'INTERA POPOLAZIONE NAZIONALE

# Ordinamento decrescente e misura del peso percentuale di ogni gruppo sui residenti
df_completo = df_completo.reset_index()
df_completo = df_completo.sort_values(by='Totale', ascending=False).reset_index(drop=True)

pop_totale = df_completo['Totale'].sum()
df_completo['quota_%'] = (df_completo['Totale'] / pop_totale) * 100
df_completo['cumulata_%'] = df_completo['quota_%'].cumsum()

# Arrotondamento all'intero e anteprima delle prime posizioni
df_completo[['Maschi', 'Femmine', 'Totale']] = df_completo[['Maschi', 'Femmine', 'Totale']].round().astype(int)
df_completo[['Stato di cittadinanza', 'Continente', 'Totale', 'quota_%', 'cumulata_%']].head(20)

,Stato di cittadinanza,Continente,Totale,quota_%,cumulata_%
0,Italia,Europa,51612485,88.028226,88.028226
1,Romania,Europa,1185655,2.022207,90.050433
2,Albania,Europa,758074,1.292941,91.343374
3,Marocco,Africa,724348,1.235419,92.578793
4,Ucraina,Europa,326790,0.557360,93.136153
5,Cina,Asia,311250,0.530856,93.667009
6,Bangladesh,Asia,281626,0.480330,94.147339
7,India,Asia,258278,0.440509,94.587848
8,Pakistan,Asia,230368,0.392907,94.980755
9,Egitto,Africa,223554,0.381285,95.362040


In [47]:
# PREPARAZIONE STRUTTURA DATI

# Copia di sicurezza e conversione numerica in float per consentire calcoli con decimali
df_completo = df_ita.copy()
for col in ['Maschi', 'Femmine', 'Totale']:
    df_completo[col] = df_completo[col].astype(float)

# Impostazione temporanea del paese come indice per velocizzare le operazioni algebriche
df_completo = df_completo.set_index('Stato di cittadinanza')

# SOTTRAZIONE DEI NATURALIZZATI DALLA VOCE ITALIA

# Scorporo dei nuovi cittadini dal gruppo autoctono mantenendo invariata la proporzione tra i sessi
totale_da_togliere = float(tot_acq['Totale'])
pm_ita = df_completo.loc['Italia', 'Maschi'] / df_completo.loc['Italia', 'Totale']

df_completo.loc['Italia', 'Totale'] -= totale_da_togliere
df_completo.loc['Italia', 'Maschi'] -= (totale_da_togliere * pm_ita)
df_completo.loc['Italia', 'Femmine'] -= (totale_da_togliere * (1 - pm_ita))

# ALLOCAZIONE DEI NATURALIZZATI AI PAESI D'ORIGINE NOTI

# Incremento delle consistenze dei singoli paesi esteri secondo le acquisizioni registrate
paesi_singoli = [c for c in col_paesi if c not in ['Totale', 'Altri paesi UE', 'Altri paesi extra-UE']]

for p in paesi_singoli:
    if p in df_completo.index and tot_acq[p] > 0:
        val = float(tot_acq[p])
        tot_attuale = df_completo.loc[p, 'Totale']
        pm = (df_completo.loc[p, 'Maschi'] / tot_attuale) if tot_attuale > 0 else 0.5
        
        df_completo.loc[p, 'Totale'] += val
        df_completo.loc[p, 'Maschi'] += val * pm
        df_completo.loc[p, 'Femmine'] += val * (1 - pm)

# Ripristino dello stato di cittadinanza come normale colonna del DataFrame
df_completo = df_completo.reset_index()


# APPLICAZIONE SOGLIA DIMENSIONALE E DIVISIONE POPOLAZIONE

# Isolamento degli italiani autoctoni e filtro dei paesi esteri sopra e sotto i 50.000 residenti
SOGLIA_MINIMA = 50_000

df_ita_solo = df_completo[df_completo['Stato di cittadinanza'] == 'Italia'].copy()
df_estero = df_completo[df_completo['Stato di cittadinanza'] != 'Italia'].copy()

mask_top = df_estero['Totale'] >= SOGLIA_MINIMA
df_top_paesi = df_estero[mask_top].copy()
df_sotto_soglia = df_estero[~mask_top].copy()

# ACCORPAMENTO CONTINENTALE DEI PAESI MINORI

# Fusione dei paesi sotto soglia nei rispettivi contenitori macro-regionali
df_continenti = df_sotto_soglia.groupby('Continente', as_index=False)[['Maschi', 'Femmine', 'Totale']].sum()
df_continenti['Stato di cittadinanza'] = 'Altro (' + df_continenti['Continente'] + ')'
df_continenti = df_continenti.set_index('Stato di cittadinanza')

# RIPARTIZIONE DELLE ACQUISIZIONI RESIDUE UE ED EXTRA-UE

# Assegnazione del residuo UE ad Altro Europa
val_ue = float(tot_acq.get('Altri paesi UE', 0))
if 'Altro (Europa)' in df_continenti.index and val_ue > 0:
    tot_eu = df_continenti.loc['Altro (Europa)', 'Totale']
    pm_eu = (df_continenti.loc['Altro (Europa)', 'Maschi'] / tot_eu) if tot_eu > 0 else 0.5
    df_continenti.loc['Altro (Europa)', 'Totale'] += val_ue
    df_continenti.loc['Altro (Europa)', 'Maschi'] += val_ue * pm_eu
    df_continenti.loc['Altro (Europa)', 'Femmine'] += val_ue * (1 - pm_eu)

# Suddivisione proporzionale del residuo extra-UE sugli altri continenti in base al loro peso
val_extra = float(tot_acq.get('Altri paesi extra-UE', 0))
continenti_extra = [c for c in ['Altro (Asia)', 'Altro (Africa)', 'Altro (America)', 'Altro (Oceania)'] 
                    if c in df_continenti.index]

if continenti_extra and val_extra > 0:
    peso_tot = df_continenti.loc[continenti_extra, 'Totale'].sum()
    for c in continenti_extra:
        quota = df_continenti.loc[c, 'Totale'] / peso_tot
        aggiunta = val_extra * quota
        tot_c = df_continenti.loc[c, 'Totale']
        pm_c = (df_continenti.loc[c, 'Maschi'] / tot_c) if tot_c > 0 else 0.5
        
        df_continenti.loc[c, 'Totale'] += aggiunta
        df_continenti.loc[c, 'Maschi'] += aggiunta * pm_c
        df_continenti.loc[c, 'Femmine'] += aggiunta * (1 - pm_c)

df_continenti = df_continenti.reset_index()


# UNIONE FINALE E CALCOLO INCIDENZA SULLA POPOLAZIONE

# Ricomposizione del dataset ordinato per dimensione e calcolo della percentuale cumulata
colonne_finali = ['Stato di cittadinanza', 'Continente', 'Maschi', 'Femmine', 'Totale']

df_finale = pd.concat([
    df_ita_solo[colonne_finali],
    df_top_paesi[colonne_finali].sort_values(by='Totale', ascending=False),
    df_continenti[colonne_finali].sort_values(by='Totale', ascending=False)
], ignore_index=True)

pop_tot = df_finale['Totale'].sum()
df_finale['quota_%'] = (df_finale['Totale'] / pop_tot) * 100
df_finale['cumulata_%'] = df_finale['quota_%'].cumsum()

# ARROTONDAMENTO E VISUALIZZAZIONE RISULTATI
# Conversione dei valori a numeri interi per la presentazione finale
for col in ['Maschi', 'Femmine', 'Totale']:
    df_finale[col] = df_finale[col].round().astype(int)

pd.set_option('display.max_rows', 50)
df_finale

,Stato di cittadinanza,Continente,Maschi,Femmine,Totale,quota_%,cumulata_%
0,Italia,Europa,25224326,26388159,51612485,87.563474,87.563474
1,Romania,Europa,518332,667323,1185655,2.011530,89.575004
2,Albania,Europa,390016,368058,758074,1.286115,90.861119
3,Marocco,Africa,397132,327216,724348,1.228897,92.090016
4,Ucraina,Europa,82014,244776,326790,0.554418,92.644434
5,Cina,Asia,156794,154456,311250,0.528053,93.172487
6,Bangladesh,Asia,208742,72884,281626,0.477794,93.650281
7,India,Asia,146186,112092,258278,0.438183,94.088464
8,Pakistan,Asia,171329,59039,230368,0.390832,94.479296
9,Egitto,Africa,152976,70578,223554,0.379272,94.858568


In [48]:
# CARICAMENTO POPOLAZIONE NAZIONALE PER ETÀ

# Lettura dei dati anagrafici per singolo anno da zero a cento anni
df_res = pd.read_csv("Popolazione residente.csv", sep=",", encoding="utf-8")
df_stran = pd.read_csv("Popolazione straniera residente.csv", sep=",", encoding="utf-8")

# STANDARDIZZAZIONE INTESTAZIONI COLONNE

# Allineamento dei nomi delle colonne per rendere omogenei i due dataset
df_res = df_res.rename(columns={
    'Età': 'Eta', 
    'Totale maschi': 'Maschi', 
    'Totale femmine': 'Femmine'
})
df_stran = df_stran.rename(columns={'Età': 'Eta'})

# PULIZIA ETÀ E GESTIONE CLASSE APERTA
# Estrazione del valore numerico per gestire correttamente la classe cento e oltre
for df_temp in [df_res, df_stran]:
    df_temp['Eta'] = df_temp['Eta'].astype(str).str.extract(r'(\d+)')[0].astype(int)
    for c in ['Maschi', 'Femmine', 'Totale']:
        df_temp[c] = pd.to_numeric(df_temp[c], errors='coerce').fillna(0)

# Ordinamento per età crescente da 0 a 100
df_res = df_res.sort_values(by='Eta').reset_index(drop=True)
df_stran = df_stran.sort_values(by='Eta').reset_index(drop=True)

# CALCOLO DELLA PIRAMIDE DEGLI ITALIANI AUTOCTONI

# Sottrazione puntuale dei residenti stranieri dal totale per ciascun anno d'età
df_autoctoni = df_res.copy()
df_autoctoni['Maschi'] = df_res['Maschi'] - df_stran['Maschi']
df_autoctoni['Femmine'] = df_res['Femmine'] - df_stran['Femmine']
df_autoctoni['Totale'] = df_autoctoni['Maschi'] + df_autoctoni['Femmine']

# VERIFICA DIFFERENZE ANAGRAFICHE

# Confronto della struttura generazionale tra popolazione autoctona e straniera
print("--- AUTOCTONI ITALIANI (Discendenti) ---")
print(df_autoctoni[df_autoctoni['Eta'].isin([0, 20, 40, 70])][['Eta', 'Maschi', 'Femmine', 'Totale']])

print("\n--- STRANIERI RESIDENTI ---")
print(df_stran[df_stran['Eta'].isin([0, 20, 40, 70])][['Eta', 'Maschi', 'Femmine', 'Totale']])

--- AUTOCTONI ITALIANI (Discendenti) ---
    Eta  Maschi  Femmine  Totale
0     0  158647   150379  309026
20   20  280637   262961  543598
40   40  286840   285155  571995
70   70  322902   352867  675769

--- STRANIERI RESIDENTI ---
    Eta  Maschi  Femmine  Totale
0     0   25636    23511   49147
20   20   33926    20845   54771
40   40   60013    56285  116298
70   70    9028    17236   26264


### Costruzione della Popolazione Base (Anno 2025)

#### Sintesi del lavoro preliminare svolto finora
Prima di far partire il motore di simulazione, abbiamo pulito e armonizzato i dati anagrafici grezzi dell'Istat attraverso quattro passaggi chiave:

1. **Filtro nazionale**: isolamento dei soli dati complessivi dell'Italia intera dal file anagrafico 2025;
2. **Rettifica etno-demografica**: scorporo di circa **1,96 milioni di persone naturalizzate** (serie 2012–2024) dal gruppo "Italia" e loro riassegnazione ai rispettivi paesi d'origine, per tracciare la popolazione di sola discendenza autoctona;
3. **Selezione dimensionale (soglia 50.000)**: mantenimento in chiaro dei principali paesi del mondo radicati sul territorio e fusione dei paesi minori nei cinque blocchi aggregati continentali (`Altro Europa`, `Altro Asia`, `Altro Africa`, `Altro America`, `Altro Oceania`);
4. **Isolamento delle piramidi dell'età**: estrazione dal registro della popolazione delle due distinte curve anagrafiche (quella senescente per gli autoctoni e quella giovane per la popolazione di origine migratoria).

---

#### Struttura del DataFrame base risultante (`df_simulazione`)
La cella successiva incrocia i 29 gruppi territoriali con tutte le 101 fasce d'età (da 0 a 100+ anni), generando la matrice di partenza per la simulazione:

* **Dimensioni**: **2.929 righe** ($29 \text{ gruppi} \times 101 \text{ età}$)
* **Popolazione totale**: **58.942.946 residenti** (quadratura esatta con la contabilità Istat)
* **Colonne**:
  * `Stato di cittadinanza`: paese d'origine o blocco continentale;
  * `Continente`: macro-area geografica di appartenenza;
  * `Eta`: singolo anno anagrafico (da `0` a `100`);
  * `Maschi`, `Femmine`, `Totale`: consistenza demografica della coorte.

In [49]:
# CALCOLO DEI PESI DEMOGRAFICI PER ETÀ E SESSO

# Normalizzazione delle piramidi dell'età per autoctoni italiani e per la popolazione di origine migratoria
peso_m_autoctoni = df_autoctoni['Maschi'] / df_autoctoni['Maschi'].sum()
peso_f_autoctoni = df_autoctoni['Femmine'] / df_autoctoni['Femmine'].sum()

peso_m_stran = df_stran['Maschi'] / df_stran['Maschi'].sum()
peso_f_stran = df_stran['Femmine'] / df_stran['Femmine'].sum()

# GENERAZIONE DELLA MATRICE BASE (PAESE X ETÀ X SESSO)
# Incrocio di ciascuna comunità con tutte le centouno fasce anagrafiche applicando la curva di riferimento
righe_simulazione = []

for _, riga_paese in df_finale.iterrows():
    paese = riga_paese['Stato di cittadinanza']
    continente = riga_paese['Continente']
    tot_maschi = riga_paese['Maschi']
    tot_femmine = riga_paese['Femmine']
    
    # Assegnazione della piramide autoctona all'Italia e di quella migratoria a tutti gli altri paesi
    if paese == 'Italia':
        pm_curva = peso_m_autoctoni
        pf_curva = peso_f_autoctoni
    else:
        pm_curva = peso_m_stran
        pf_curva = peso_f_stran
    
    # Calcolo dei residenti maschi e femmine per ogni singolo anno da zero a cento
    for eta in range(101):
        m_eta = tot_maschi * pm_curva[eta]
        f_eta = tot_femmine * pf_curva[eta]
        
        righe_simulazione.append({
            'Stato di cittadinanza': paese,
            'Continente': continente,
            'Eta': eta,
            'Maschi': m_eta,
            'Femmine': f_eta,
            'Totale': m_eta + f_eta
        })

# CREAZIONE E ARROTONDAMENTO DEL DATAFRAME
# Strutturazione della tabella base di partenza della simulazione e arrotondamento all'unità
df_simulazione = pd.DataFrame(righe_simulazione)

for col in ['Maschi', 'Femmine', 'Totale']:
    df_simulazione[col] = df_simulazione[col].round().astype(int)

# VERIFICA DELLA COERENZA DEI TOTALI

# Controllo della quadratura della popolazione nazionale e visualizzazione di una comunità campione
print(f"Righe totali create: {len(df_simulazione)} ({df_simulazione['Stato di cittadinanza'].nunique()} gruppi × 101 fasce d'età)")
print(f"Popolazione totale nazionale: {df_simulazione['Totale'].sum():,}".replace(',', '.'))

print("\nEsempio: Distribuzione per età del Marocco:")
print(df_simulazione[(df_simulazione['Stato di cittadinanza'] == 'Marocco') & 
                     (df_simulazione['Eta'].isin([0, 10, 20, 30, 40, 50, 60, 70]))])

Righe totali create: 2929 (29 gruppi × 101 fasce d'età)
Popolazione totale nazionale: 58.942.946

Esempio: Distribuzione per età del Marocco:
    Stato di cittadinanza Continente  Eta  Maschi  Femmine  Totale
303               Marocco     Africa    0    3613     2806    6419
313               Marocco     Africa   10    4519     3476    7995
323               Marocco     Africa   20    4781     2488    7269
333               Marocco     Africa   30    8378     5191   13569
343               Marocco     Africa   40    8457     6718   15175
353               Marocco     Africa   50    6358     5973   12332
363               Marocco     Africa   60    3240     3938    7178
373               Marocco     Africa   70    1272     2057    3330


In [50]:
# CARICAMENTO E CONTROLLO DEL FILE DECESSI

# Rilevamento automatico del separatore (, oppure ;)
try:
    df_dec_raw = pd.read_csv("Decessi2025.csv", sep=";", encoding="utf-8")
    if len(df_dec_raw.columns) == 1:
        df_dec_raw = pd.read_csv("Decessi2025.csv", sep=",", encoding="utf-8")
except Exception:
    df_dec_raw = pd.read_csv("Decessi2025.csv", sep=",", encoding="utf-8")

df_dec_raw.columns = df_dec_raw.columns.astype(str).str.strip()

#print("Colonne trovate in Decessi2025.csv:")
#print(df_dec_raw.columns.tolist())
#print("\nPrime 5 righe:")
#print(df_dec_raw.head())

In [51]:
# PULIZIA INTESTAZIONI E FILTRO NAZIONALE TAVOLA DI MORTALITÀ

# Rimozione di eventuali spazi nascosti nelle colonne e isolamento del livello nazionale
df_dec_raw.columns = df_dec_raw.columns.astype(str).str.strip()
df_mortalita_ita = df_dec_raw[df_dec_raw['Ripartizione'].astype(str).str.strip().str.lower() == 'italia'].copy()


# RICONOSCIMENTO AUTOMATICO COLONNE E TASSO DI SOPRAVVIVENZA

# Rilevamento dinamico dei campi età e sopravvivenza o calcolo alternativo dalla probabilità di morte
col_sopr_candidates = [c for c in df_mortalita_ita.columns if 'sopravvivenza' in c.lower()]
col_morte_candidates = [c for c in df_mortalita_ita.columns if 'morte' in c.lower()]

col_eta = [c for c in df_mortalita_ita.columns if 'eta' in c.lower() or 'età' in c.lower()][0]
df_mortalita_ita[col_eta] = pd.to_numeric(df_mortalita_ita[col_eta], errors='coerce').fillna(0).astype(int)

if col_sopr_candidates:
    col_usata = col_sopr_candidates[0]
    df_mortalita_ita['tasso_sopravvivenza'] = pd.to_numeric(df_mortalita_ita[col_usata], errors='coerce')
else:
    col_morte = col_morte_candidates[0]
    df_mortalita_ita['tasso_sopravvivenza'] = 1.0 - (pd.to_numeric(df_mortalita_ita[col_morte], errors='coerce') / 1000.0)


# CREAZIONE DELLE SERIE DI SOPRAVVIVENZA PER SESSO

# Separazione delle probabilità attuariali per maschi e femmine indicizzate per età da zero a cento
df_m_sopr = df_mortalita_ita[df_mortalita_ita['Sesso'].astype(str).str.strip().str.lower() == 'maschi']
df_m_sopr = df_m_sopr.drop_duplicates(subset=[col_eta]).set_index(col_eta)['tasso_sopravvivenza']

df_f_sopr = df_mortalita_ita[df_mortalita_ita['Sesso'].astype(str).str.strip().str.lower() == 'femmine']
df_f_sopr = df_f_sopr.drop_duplicates(subset=[col_eta]).set_index(col_eta)['tasso_sopravvivenza']


# VERIFICA DELLA PROBABILITÀ DI SOPRAVVIVERE ALLE ETÀ CHIAVE

# Quadro di controllo per verificare la coerenza della curva di longevità per entrambi i sessi
confronto_sopr = pd.DataFrame({
    'Sopravvivenza Maschi': df_m_sopr,
    'Sopravvivenza Femmine': df_f_sopr
})

print("--- Probabilità di sopravvivere all'anno successivo per età ---")
print(confronto_sopr.loc[[0, 20, 50, 70, 80, 90]])

--- Probabilità di sopravvivere all'anno successivo per età ---
     Sopravvivenza Maschi  Sopravvivenza Femmine
Età                                             
0                0.999745               0.999757
20               0.999622               0.999853
50               0.997581               0.998595
70               0.984258               0.990909
80               0.952910               0.968108
90               0.840820               0.872157


In [52]:
# CARICAMENTO DEL FILE DELLE NASCITE

# Lettura flessibile del dataset Istat con rilevamento del separatore e pulizia spazi
try:
    df_nasc_raw = pd.read_csv("Nascite2024.csv", sep=";", encoding="utf-8")
    if len(df_nasc_raw.columns) == 1:
        df_nasc_raw = pd.read_csv("Nascite2024.csv", sep=",", encoding="utf-8")
except Exception:
    df_nasc_raw = pd.read_csv("Nascite2024.csv", sep=",", encoding="utf-8")

df_nasc_raw.columns = df_nasc_raw.columns.astype(str).str.strip()

In [53]:
# MAPPA CONTINENTI E PULIZIA REGISTRO NASCITE

# Riconciliazione geografica tra stati e continenti e normalizzazione dei nati registrati
mappa_continente = df_ita.drop_duplicates(subset=['Stato di cittadinanza']).set_index('Stato di cittadinanza')['Continente'].to_dict()

df_nasc = df_nasc_raw.copy()
df_nasc['Territorio estero'] = df_nasc['Territorio estero'].astype(str).str.strip()
df_nasc['Nascite_Italia'] = pd.to_numeric(df_nasc['Italia'], errors='coerce').fillna(0)
mappa_nascite = df_nasc.set_index('Territorio estero')['Nascite_Italia'].to_dict()


# CALCOLO DONNE IN ETÀ FERTILE (15-49 ANNI)
# Conteggio della popolazione femminile riproduttiva presente nella matrice di base per ogni gruppo
donne_15_49 = df_simulazione[(df_simulazione['Eta'] >= 15) & (df_simulazione['Eta'] <= 49)]
tot_donne_fertili = donne_15_49.groupby('Stato di cittadinanza')['Femmine'].sum()


# ASSEGNAZIONE NASCITE E CALCOLO TASSO DI FECONDITÀ ANNUO
# Determinazione delle nascite annue per comunità e calcolo del tasso per donna fertile
dati_fecondita = []
nascite_autoctoni = int(df_autoctoni.loc[df_autoctoni['Eta'] == 0, 'Totale'].values[0])

for _, riga in df_finale.iterrows():
    paese = riga['Stato di cittadinanza']
    continente = riga['Continente']
    n_donne = tot_donne_fertili.get(paese, 0)
    
    # Assegnazione per autoctoni italiani
    if paese == 'Italia':
        nascite = nascite_autoctoni
    
    # Assegnazione per i singoli paesi esteri con riga dedicata
    elif paese in mappa_nascite:
        nascite = mappa_nascite[paese]
        
    # Assegnazione per i blocchi continentali sommando i paesi minori esclusi dai top
    else:
        paesi_top = df_finale[~df_finale['Stato di cittadinanza'].str.startswith('Altro')]['Stato di cittadinanza'].tolist()
        nascite = sum(
            val for p, val in mappa_nascite.items() 
            if mappa_continente.get(p) == continente and p not in paesi_top
        )
    
    # Rapporto annuo tra nuovi nati e platea femminile fertile
    tasso = (nascite / n_donne) if n_donne > 0 else 0.035
    
    dati_fecondita.append({
        'Stato di cittadinanza': paese,
        'Continente': continente,
        'Donne_15_49': int(n_donne),
        'Nascite_Annue': int(nascite),
        'Tasso_Fecondita_Annuo': round(tasso, 4)
    })

df_fecondita = pd.DataFrame(dati_fecondita)


# PARAMETRI BIOLOGICI DI GENERE ALLA NASCITA
# Definizione della quota maschile e femminile naturale ricavata dai neonati a età zero
PROP_M_NASCITA = 184283 / 358173  
PROP_F_NASCITA = 1 - PROP_M_NASCITA 

print("--- TASSI DI FECONDITÀ PER PAESE D'ORIGINE ---")
print(df_fecondita.head(15))

--- TASSI DI FECONDITÀ PER PAESE D'ORIGINE ---
   Stato di cittadinanza Continente  Donne_15_49  Nascite_Annue  Tasso_Fecondita_Annuo
0                 Italia     Europa      9476195         309026                 0.0326
1                Romania     Europa       361912           7188                 0.0199
2                Albania     Europa       199612           5570                 0.0279
3                Marocco     Africa       177458           5708                 0.0322
4                Ucraina     Europa       132748            859                 0.0065
5                   Cina       Asia        83768            767                 0.0092
6             Bangladesh       Asia        39527           3785                 0.0958
7                  India       Asia        60789           2479                 0.0408
8               Pakistan       Asia        32017           2287                 0.0714
9                 Egitto     Africa        38277           2570                 0.0

In [54]:
# CARICAMENTO DEI FILE DI FLUSSO MIGRATORIO
# Lettura delle tavole Istat su immigrazione, emigrazione e mobilità internazionale
df_imm = pd.read_excel("immigrati_per_paese.xlsx", header=1)
df_emig = pd.read_excel("emigrati_per_paese.xlsx", header=1)
df_migr_ita = pd.read_excel("migrazioni_italiani.xlsx", header=1)

# NORMALIZZAZIONE INTESTAZIONI E PULIZIA DEI NOMI DEI PAESI
# Ridenominazione del campo geografico e rimozione di spazi superflui
for df_temp in [df_imm, df_emig, df_migr_ita]:
    df_temp.rename(columns={df_temp.columns[0]: 'Paese'}, inplace=True)
    df_temp.dropna(subset=['Paese'], inplace=True)
    df_temp['Paese'] = df_temp['Paese'].astype(str).str.strip()

# RILEVAMENTO ANNI DISPONIBILI NELLA SERIE STORICA
# Estrazione delle colonne temporali numeriche presenti nelle tavole
anni_disponibili = [c for c in df_imm.columns if str(c).replace('.0','').isdigit()]
print("Anni disponibili nei dati migratori:", [int(a) for a in anni_disponibili[-5:]])

# ANTEPRIMA FLUSSI DI INGRESSO RECENTI
# Visualizzazione delle principali voci di immigrazione nell'ultimo anno censito
anno_recente = anni_disponibili[-1]
print(f"\n--- Top 5 paesi per Immigrazione nell'anno {int(anno_recente)} ---")
print(df_imm[['Paese', anno_recente]].head(8))

# ANTEPRIMA FLUSSI COMPLESSIVI ITALIANI
# Verifica della struttura del bilancio migratorio con l'estero
print(f"\n--- Migrazioni Italiani (prime righe) ---")
print(df_migr_ita.head(5))

Anni disponibili nei dati migratori: [2021, 2022, 2023, 2024, 2025]

--- Top 5 paesi per Immigrazione nell'anno 2025 ---
                    Paese     2025
0                ISCRITTI      NaN
1                  Europa  97412.0
2             EU27 (2020)  36749.0
3       Extra EU27 (2020)  60663.0
4        EU28 (2007-2020)  38035.0
5  Extra EU28 (2007-2020)  59377.0
7                 Francia   2592.0
8                Germania   2937.0

--- Migrazioni Italiani (prime righe) ---
                          Paese     2002     2003      2004     2005     2006     2007     2008     2009     2010     2011     2012     2013     2014     2015     2016     2017     2018     2019     2020     2021     2022     2023     2024     2025
0                      ISCRITTI      NaN      NaN       NaN      NaN      NaN      NaN      NaN      NaN      NaN      NaN      NaN      NaN      NaN      NaN      NaN      NaN      NaN      NaN      NaN      NaN      NaN      NaN      NaN      NaN
1                      

In [55]:
# VERIFICA STRUTTURA DEI FOGLI EXCEL
# Controllo della presenza di fogli multipli nelle cartelle di lavoro
excel_migr_ita = pd.ExcelFile("migrazioni_italiani.xlsx")
excel_emig = pd.ExcelFile("emigrati_per_paese.xlsx")

print("Fogli in migrazioni_italiani.xlsx:", excel_migr_ita.sheet_names)
print("Fogli in emigrati_per_paese.xlsx:", excel_emig.sheet_names)

# IDENTIFICAZIONE DEI TOTALI NAZIONALI DI ISCRIZIONE E CANCELLAZIONE
# Ricerca delle righe di riepilogo nazionale per quantificare espatri e rimpatri
voci_interessanti = df_migr_ita[df_migr_ita['Paese'].str.lower().str.contains('italia|totale|iscritti|cancellati', na=False)]
print("\nRighe chiave trovate in migrazioni_italiani.xlsx:")
print(voci_interessanti[['Paese', 2023, 2024]])

# ANTEPRIMA CANCELLAZIONI PER L'ESTERO
# Controllo dei valori di emigrazione per destinazione geografica
print("\n--- Anteprima emigrati_per_paese.xlsx ---")
print(df_emig[['Paese', 2023, 2024]].head(6))

Fogli in migrazioni_italiani.xlsx: ['Tavola 3']
Fogli in emigrati_per_paese.xlsx: ['tavola 8']

Righe chiave trovate in migrazioni_italiani.xlsx:
         Paese      2023      2024
0     ISCRITTI       NaN       NaN
23      Italia  439658.0  451583.0
24  CANCELLATI       NaN       NaN
47      Italia  158438.0  188903.0

--- Anteprima emigrati_per_paese.xlsx ---
                    Paese     2023     2024
0              CANCELLATI      NaN      NaN
1                  Europa  26381.0  31062.0
2             EU27 (2020)  16867.0  21357.0
3       Extra EU27 (2020)   9514.0   9705.0
4        EU28 (2007-2020)  17409.0  21777.0
5  Extra EU28 (2007-2020)   8972.0   9285.0


### Determinazione dei Flussi Migratori e del Saldo Netto Annuo

#### Obiettivo metodologico
In questo passaggio calcoliamo il **Saldo Migratorio Netto Annuo** ($\text{Saldo Netto} = \text{Immigrati} - \text{Emigrati}$) per ciascuna comunità presente nel nostro modello (popolazione autoctona italiana, singoli paesi di origine principali e blocchi aggregati continentali).  
Poiché il saldo naturale italiano (nascite meno decessi) è stabilmente negativo di oltre 300.000 unità all'anno, la componente migratoria rappresenta il motore dinamico più determinante nell'evoluzione demografica di medio-lungo periodo.

---

#### Fonti dei Dati Utilizzate
Per garantire la coerenza complessiva dei flussi, utilizziamo tre tavole ufficiali Istat con anno di riferimento **2024**:
1. **`migrazioni_italiani.xlsx` (Tavola 3 Istat)**: contiene il quadro riassuntivo nazionale di *tutte* le iscrizioni anagrafiche dall'estero (451.583 persone nel 2024) e di *tutte* le cancellazioni per l'estero (188.903 persone nel 2024).
2. **`immigrati_per_paese.xlsx` (Tavola 7 Istat)**: registra le iscrizioni dall'estero dei soli cittadini stranieri, dettagliate per macro-continenti e per i principali paesi di provenienza.
3. **`emigrati_per_paese.xlsx` (Tavola 8 Istat)**: registra le cancellazioni anagrafiche per l'estero dei soli cittadini stranieri.

---

#### Logica di Calcolo e Procedimento

1. **Chiusura contabile per la popolazione autoctona (Italia)**:  
   L'Istat non pubblica una tavola separata per i soli espatri/rimpatri degli italiani autoctoni. Ricaviamo i loro flussi per sottrazione esatta:
   $$\text{Immigrati}_{\text{Italia}} = \text{Iscritti Totali Nazionali} - \sum \text{Iscritti Stranieri}$$
   $$\text{Emigrati}_{\text{Italia}} = \text{Cancellati Totali Nazionali} - \sum \text{Cancellati Stranieri}$$
   Questo permette di quantificare sia il fenomeno dell'emigrazione dei cittadini italiani all'estero sia il loro tasso di rimpatrio.

2. **Assegnazione diretta per i Paesi Top**:  
   Per tutti i paesi esplicitati nominativamente nelle Tavole 7 e 8 (es. *Romania, Marocco, Albania, Cina, Bangladesh, India, Pakistan, Egitto, Tunisia, Senegal, Perù, Nigeria*), vengono presi i valori puntuali registrati dall'Istat per l'anno di riferimento.

3. **Calcolo dei residui continentali**:  
   Nelle tabelle dell'Istat sono presenti le righe di totale per continente (*Asia, Africa, Europa, America, Oceania*), ma sono elencati singolarmente solo i 20-25 paesi con i flussi più voluminosi. Il flusso dei paesi minori appartenenti a quel continente viene calcolato come differenza:
   $$\text{Residuo Immigrati}_{C} = \text{Totale Continente}_{C} - \sum_{p \in C \text{ (dettagliati)}} \text{Immigrati}_{p}$$
   $$\text{Residuo Emigrati}_{C} = \text{Totale Continente}_{C} - \sum_{p \in C \text{ (dettagliati)}} \text{Emigrati}_{p}$$

4. **Trattamento delle Filippine e dei blocchi continentali `Altro (...)`**:  
   La comunità filippina ha uno stock residente elevato (oltre 150.000 persone, rientrando nei nostri paesi Top), ma flussi recenti inferiori a quelli di Bangladesh o Pakistan, motivo per cui l'Istat non la scorpora dal totale dell'Asia.  
   * Il residuo asiatico viene quindi ripartito tra le **Filippine** e la categoria **`Altro (Asia)`** in proporzione alla popolazione residente già stimata in `df_finale`.
   * Per tutti gli altri continenti (*Africa, America, Oceania, Europa*), il residuo viene interamente attribuito ai rispettivi gruppi aggregati (`Altro (Africa)`, `Altro (America)`, ecc.).

---

#### Approssimazioni Metodologiche Adottate

* **Ipotesi di stazionarietà dei flussi (*Static Flow Assumption*)**:  
  Nel modello base proiettiamo i volumi migratori mantenendoli costanti al livello osservato nel 2024. È la metodologia standard adottata nei cosiddetti "scenari a politiche invariate" (*baseline projection*).
* **Ripartizione proporzionale dello stock residuo**:  
  La suddivisione del residuo asiatico sulle Filippine presuppone che l'intensità migratoria recente sia proporzionale alla consistenza numerica preesistente della comunità sul territorio nazionale.
* **Conservazione esatta dei totali nazionali (*Closure Constraint*)**:  
  Grazie a questa quadratura, la somma di tutti gli immigrati e di tutti gli emigrati della tabella finale coincide **all'unità** con i totali ufficiali dell'Istat, evitando qualsiasi dispersione o sovrastima di individui nella simulazione.

In [60]:
# PARAMETRI TEMPORALI E PULIZIA ARCHIVI MIGRATORI
# Selezione dell'anno di riferimento ed eliminazione di spazi residui nei nomi dei paesi
ANNO_FLUSSI = 2024

df_imm_clean = df_imm.copy()
df_emig_clean = df_emig.copy()

df_imm_clean['Paese'] = df_imm_clean['Paese'].astype(str).str.strip()
df_emig_clean['Paese'] = df_emig_clean['Paese'].astype(str).str.strip()

imm_dict = df_imm_clean.set_index('Paese')[ANNO_FLUSSI].dropna().to_dict()
emig_dict = df_emig_clean.set_index('Paese')[ANNO_FLUSSI].dropna().to_dict()


# CALCOLO DEI FLUSSI PER GLI ITALIANI AUTOCTONI
# Sottrazione del totale degli stranieri dal dato nazionale complessivo di ingressi e uscite
tot_iscritti_nazionale = float(df_migr_ita.loc[23, ANNO_FLUSSI])
tot_cancellati_nazionale = float(df_migr_ita.loc[47, ANNO_FLUSSI])

continenti_list = ['Europa', 'Africa', 'Asia', 'America', 'Oceania']
tot_imm_stranieri = sum(float(imm_dict.get(c, 0)) for c in continenti_list)
tot_emig_stranieri = sum(float(emig_dict.get(c, 0)) for c in continenti_list)

imm_italiani = max(0.0, tot_iscritti_nazionale - tot_imm_stranieri)
emig_italiani = max(0.0, tot_cancellati_nazionale - tot_emig_stranieri)


# CALCOLO DEI FLUSSI RESIDUI CONTINENTALI
# Differenza tra il totale continentale e la somma dei paesi con riga dedicata
residuo_imm = {}
residuo_emig = {}

for cont in continenti_list:
    tot_c_imm = float(imm_dict.get(cont, 0))
    tot_c_emig = float(emig_dict.get(cont, 0))
    
    paesi_singoli_cont = [p for p in df_finale['Stato di cittadinanza'] 
                          if mappa_continente.get(p) == cont and p in imm_dict and not p.startswith('Altro')]
    
    imm_spiegati = sum(float(imm_dict.get(p, 0)) for p in paesi_singoli_cont)
    emig_spiegati = sum(float(emig_dict.get(p, 0)) for p in paesi_singoli_cont)
    
    residuo_imm[cont] = max(0.0, tot_c_imm - imm_spiegati)
    residuo_emig[cont] = max(0.0, tot_c_emig - emig_spiegati)


# RILEVAMENTO GRUPPI DIPENDENTI DAL RESIDUO
# Mappatura dei paesi e blocchi aggregati non esplicitati nominativamente nelle tavole Istat
gruppi_residuo_per_cont = {}
for cont in continenti_list:
    gruppi_residuo_per_cont[cont] = [
        p for p in df_finale['Stato di cittadinanza'] 
        if mappa_continente.get(p) == cont and (p not in imm_dict or p.startswith('Altro')) and p != 'Italia'
    ]


# ASSEGNAZIONE DEI FLUSSI E DETERMINAZIONE DEL SALDO NETTO ANNUO
# Calcolo di immigrati, emigrati e saldo netto distribuendo i residui in base al peso demografico
dati_migr = []

for _, riga in df_finale.iterrows():
    paese = riga['Stato di cittadinanza']
    continente = riga['Continente']
    
    # Assegnazione flussi per autoctoni italiani
    if paese == 'Italia':
        imm = imm_italiani
        emig = emig_italiani
        
    # Assegnazione per paesi esteri esplicitati nel dataset
    elif paese in imm_dict and not paese.startswith('Altro'):
        imm = float(imm_dict.get(paese, 0))
        emig = float(emig_dict.get(paese, 0))
        
    # Ripartizione proporzionale del residuo continentale
    else:
        gruppi_cont = gruppi_residuo_per_cont.get(continente, [])
        pop_tot_gruppi = df_finale[df_finale['Stato di cittadinanza'].isin(gruppi_cont)]['Totale'].sum()
        pop_questo = riga['Totale']
        
        quota = (pop_questo / pop_tot_gruppi) if pop_tot_gruppi > 0 else 0
        imm = residuo_imm.get(continente, 0) * quota
        emig = residuo_emig.get(continente, 0) * quota
            
    dati_migr.append({
        'Stato di cittadinanza': paese,
        'Continente': continente,
        'Immigrati_Annui': int(round(imm)),
        'Emigrati_Annui': int(round(emig)),
        'Saldo_Netto_Annuo': int(round(imm - emig))
    })

df_migrazioni = pd.DataFrame(dati_migr)

print("--- SALDO MIGRATORIO ---")
print(df_migrazioni.head(20))

--- SALDO MIGRATORIO ---
   Stato di cittadinanza Continente  Immigrati_Annui  Emigrati_Annui  Saldo_Netto_Annuo
0                 Italia     Europa            58507          141057             -82550
1                Romania     Europa            23148           15335               7813
2                Albania     Europa            27587            1720              25867
3                Marocco     Africa            24488            1561              22927
4                Ucraina     Europa            27861            6444              21417
5                   Cina       Asia             7235            2443               4792
6             Bangladesh       Asia            30733             826              29907
7                  India       Asia            13791            1462              12329
8               Pakistan       Asia            21407            1198              20209
9                 Egitto     Africa            20853             720              20133
10     

In [65]:
# PARAMETRI CONFIGURABILI DELLA PROIEZIONE
# Definizione dell'orizzonte temporale e della finestra anagrafica dei flussi migratori
ANNO_INIZIO = 2025
ANNO_FINE = 2200
ETA_MIN_MIGRANTI = 18
ETA_MAX_MIGRANTI = 38
ETA_PICCO_MIGRANTI = 26


# VETTORI E PARAMETRI ATTUARIALI DI BASE
# Estrazione delle probabilità di sopravvivenza, fecondità, saldi e quote di genere
sopr_m = np.array([df_m_sopr.get(eta, 0.5) for eta in range(101)])
sopr_f = np.array([df_f_sopr.get(eta, 0.5) for eta in range(101)])

tassi_fecondita = df_fecondita.set_index('Stato di cittadinanza')['Tasso_Fecondita_Annuo'].to_dict()
saldi_migratori = df_migrazioni.set_index('Stato di cittadinanza')['Saldo_Netto_Annuo'].to_dict()
prop_m_paese = (df_finale.set_index('Stato di cittadinanza')['Maschi'] / df_finale.set_index('Stato di cittadinanza')['Totale']).to_dict()


# MODELLAZIONE DELLA CURVA DI ETÀ DEI MIGRANTI
# Costruzione della distribuzione a campana concentrata tra diciotto e trentotto anni
pesi_eta_migr = np.zeros(101)
x_migr = np.arange(ETA_MIN_MIGRANTI, ETA_MAX_MIGRANTI + 1)
campana = np.exp(-0.5 * ((x_migr - ETA_PICCO_MIGRANTI) / 4.5) ** 2)
pesi_eta_migr[ETA_MIN_MIGRANTI:ETA_MAX_MIGRANTI + 1] = campana / campana.sum()


# INIZIALIZZAZIONE DELLA SERIE DI SIMULAZIONE
# Registrazione della popolazione di partenza all'anno base e mappatura geografica
lista_anni_popolazione = []
df_anno_corrente = df_simulazione.copy()
df_anno_corrente['Anno'] = ANNO_INIZIO
lista_anni_popolazione.append(df_anno_corrente)

mappa_cont_finale = df_finale.set_index('Stato di cittadinanza')['Continente'].to_dict()
elenco_paesi = df_finale['Stato di cittadinanza'].unique()

print(f"Avvio simulazione demografica dal {ANNO_INIZIO} al {ANNO_FINE}...")


# CICLO DI PROIEZIONE DEMOGRAFICA ANNUALE
# Avanzamento della popolazione anno per anno applicando mortalità, natalità e migrazioni
for anno in range(ANNO_INIZIO, ANNO_FINE):
    righe_nuovo_anno = []
    
    for paese in elenco_paesi:
        continente = mappa_cont_finale[paese]
        
        sub_df = df_anno_corrente[df_anno_corrente['Stato di cittadinanza'] == paese].sort_values(by='Eta')
        m_curr = sub_df['Maschi'].values.astype(float)
        f_curr = sub_df['Femmine'].values.astype(float)
        
        # INVECCHIAMENTO E APPLICAZIONE DELLA MORTALITÀ
        # Avanzamento di un anno anagrafico moltiplicando per il rispettivo tasso di sopravvivenza
        m_next = np.zeros(101)
        f_next = np.zeros(101)
        
        m_next[1:101] = m_curr[0:100] * sopr_m[0:100]
        f_next[1:101] = f_curr[0:100] * sopr_f[0:100]
        
        m_next[100] += m_curr[100] * sopr_m[100]
        f_next[100] += f_curr[100] * sopr_f[100]
        
        # CALCOLO DELLE NASCITE E ASSEGNAZIONE ALLA COORTE ZERO
        # Generazione dei nuovi nati dalle donne tra quindici e quarantanove anni
        donne_fertili = f_curr[15:50].sum()
        tasso_fec = tassi_fecondita.get(paese, 0.033)
        nascite_anno = donne_fertili * tasso_fec
        
        m_next[0] = nascite_anno * PROP_M_NASCITA
        f_next[0] = nascite_anno * PROP_F_NASCITA
        
        # INTEGRAZIONE DEI SALDI MIGRATORI PER GENERE ED ETÀ
        # Aggiunta dei flussi netti secondo la sex-ratio propria del paese e la curva lavorativa
        saldo = saldi_migratori.get(paese, 0)
        quota_m = prop_m_paese.get(paese, 0.5)
        quota_f = 1.0 - quota_m
        
        m_next += (saldo * quota_m) * pesi_eta_migr
        f_next += (saldo * quota_f) * pesi_eta_migr
        
        m_next = np.maximum(0.0, m_next)
        f_next = np.maximum(0.0, f_next)
        
        # REGISTRAZIONE DATI COORTE PER QUESTO PAESE
        for eta in range(101):
            righe_nuovo_anno.append({
                'Anno': anno + 1,
                'Stato di cittadinanza': paese,
                'Continente': continente,
                'Eta': eta,
                'Maschi': m_next[eta],
                'Femmine': f_next[eta],
                'Totale': m_next[eta] + f_next[eta]
            })
            
    df_anno_corrente = pd.DataFrame(righe_nuovo_anno)
    lista_anni_popolazione.append(df_anno_corrente)


# COSTRUZIONE DEL DATAFRAME LONGITUDINALE COMPLETO
# Concatenazione di tutte le annate simulate e arrotondamento all'intero
df_proiezione = pd.concat(lista_anni_popolazione, ignore_index=True)

for col in ['Maschi', 'Femmine', 'Totale']:
    df_proiezione[col] = df_proiezione[col].round().astype(int)

df_proiezione = df_proiezione[['Anno', 'Stato di cittadinanza', 'Continente', 'Eta', 'Maschi', 'Femmine', 'Totale']]

print("Simulazione completata con successo!")
print(f"Dimensioni dataset finale: {df_proiezione.shape[0]:,} righe ({ANNO_FINE - ANNO_INIZIO + 1} anni simulati)")

Avvio simulazione demografica dal 2025 al 2200...
Simulazione completata con successo!
Dimensioni dataset finale: 515,504 righe (176 anni simulati)


In [66]:
from IPython.display import display

# EVOLUZIONE DELLA POPOLAZIONE TOTALE NAZIONALE
# Calcolo delle consistenze complessive negli anni cardine e formattazione dei volumi
totali_per_anno = df_proiezione.groupby('Anno')['Totale'].sum()
anni_chiave = [ANNO_INIZIO, 2030, 2040, ANNO_FINE]

df_sintesi_totale = pd.DataFrame({
    'Anno': anni_chiave,
    'Popolazione Totale': [totali_per_anno[a] for a in anni_chiave]
})

df_sintesi_totale['Popolazione'] = df_sintesi_totale['Popolazione Totale'].apply(lambda x: f"{int(x):,}".replace(",", "."))
df_sintesi_totale['Differenza dal 2025'] = (df_sintesi_totale['Popolazione Totale'] - totali_per_anno[ANNO_INIZIO]).apply(lambda x: f"{int(x):+,}".replace(",", "."))

print("=== EVOLUZIONE POPOLAZIONE RESIDENTE IN ITALIA ===")
display(df_sintesi_totale[['Anno', 'Popolazione', 'Differenza dal 2025']])


# CONFRONTO PER COMUNITÀ D'ORIGINE (TOP VENTI)
# Calcolo delle quote percentuali sul totale nazionale, delle variazioni assolute e della crescita relativa
confronto_origini = df_proiezione.groupby(['Anno', 'Stato di cittadinanza'])['Totale'].sum().unstack(level=0)

tot_inizio = totali_per_anno[ANNO_INIZIO]
tot_fine = totali_per_anno[ANNO_FINE]

confronto_origini[f'Quota % {ANNO_INIZIO}'] = (confronto_origini[ANNO_INIZIO] / tot_inizio) * 100
confronto_origini[f'Quota % {ANNO_FINE}'] = (confronto_origini[ANNO_FINE] / tot_fine) * 100
confronto_origini['Variazione Assoluta'] = confronto_origini[ANNO_FINE] - confronto_origini[ANNO_INIZIO]
confronto_origini['Crescita %'] = (confronto_origini['Variazione Assoluta'] / confronto_origini[ANNO_INIZIO]) * 100

top20 = confronto_origini.sort_values(by=ANNO_INIZIO, ascending=False).head(20).copy()


# FORMATTAZIONE E PRESENTAZIONE TABELLARE DEI RISULTATI
# Generazione del prospetto comparativo finale formattato con separatori delle migliaia e percentuali
tabella_pulita = pd.DataFrame(index=top20.index)
tabella_pulita[f'Popolazione {ANNO_INIZIO}'] = top20[ANNO_INIZIO].apply(lambda x: f"{int(x):,}".replace(",", "."))
tabella_pulita[f'Quota {ANNO_INIZIO}'] = top20[f'Quota % {ANNO_INIZIO}'].apply(lambda x: f"{x:.2f}%")
tabella_pulita[f'Popolazione {ANNO_FINE}'] = top20[ANNO_FINE].apply(lambda x: f"{int(x):,}".replace(",", "."))
tabella_pulita[f'Quota {ANNO_FINE}'] = top20[f'Quota % {ANNO_FINE}'].apply(lambda x: f"{x:.2f}%")
tabella_pulita['Differenza Persone'] = top20['Variazione Assoluta'].apply(lambda x: f"{int(x):+,}".replace(",", "."))
tabella_pulita['Crescita Gruppo %'] = top20['Crescita %'].apply(lambda x: f"{x:+.1f}%")

print(f"\n=== CONFRONTO {ANNO_INIZIO} vs {ANNO_FINE} PER ORIGINE (Top 20) ===")
display(tabella_pulita)

=== EVOLUZIONE POPOLAZIONE RESIDENTE IN ITALIA ===


,Anno,Popolazione,Differenza dal 2025
0,2025,58.942.946,+0
1,2030,58.906.875,-36.071
2,2040,57.984.476,-958.470
3,2200,67.995.530,+9.052.584



=== CONFRONTO 2025 vs 2200 PER ORIGINE (Top 20) ===


,Popolazione 2025,Quota 2025,Popolazione 2200,Quota 2200,Differenza Persone,Crescita Gruppo %
Stato di cittadinanza,,,,,,
Italia,51.612.483,87.56%,0,0.00%,-51.612.483,-100.0%
Romania,1.185.656,2.01%,726.936,1.07%,-458.720,-38.7%
Albania,758.075,1.29%,2.766.978,4.07%,+2.008.903,+265.0%
Marocco,724.346,1.23%,2.684.571,3.95%,+1.960.225,+270.6%
Altro (Europa),448.446,0.76%,2.145.547,3.16%,+1.697.101,+378.4%
Altro (Africa),360.159,0.61%,10.830,0.02%,-349.329,-97.0%
Ucraina,326.790,0.55%,1.494.125,2.20%,+1.167.335,+357.2%
Cina,311.246,0.53%,329.248,0.48%,+18.002,+5.8%
Bangladesh,281.623,0.48%,21.006.978,30.89%,+20.725.355,+7359.3%
